# Quick guide to `emgapi_v2_client` 
`mgnipy.emgapi_v2_client` is a python library submodule that was created from the [openapi.json spec](ttps://www.ebi.ac.uk/metagenomics/api/v2/openapi.json) using [openapi-python-client](https://github.com/openapi-generators/openapi-python-client).

## Introduction 
The purpose of this page is to better understand the python client libraries generated using `openapi-python-client` -- in case you want to work with the `mgnipy.emgapi_v2_client` submodule directly.

<details>
<summary style=color:green>
As stated in <b>README_emgapi-v2-client.md</b> - Things to know:
</summary>
<h1></h1>

1. Every path/method combo becomes a Python module with four functions:
   1. `sync`: Blocking request that returns parsed data (if successful) or `None`
   1. `sync_detailed`: Blocking request that always returns a `Request`, optionally with `parsed` set if the request was successful.
   1. `asyncio`: Like `sync` but async instead of blocking
   1. `asyncio_detailed`: Like `sync_detailed` but async instead of blocking

1. All path/query params, and bodies become method arguments.
1. If your endpoint had any tags on it, the first tag will be used as a module name for the function (my_tag above)

<h1></h1>
</details>
<br>

Below we will have an example where we will find studies of the biome root:Host-associated:Plants

-----

## Required imports

At a minimum:

1. We need to find and import the appropriate module for our endpoint: [`/Miscellaneous/list_mgnify_biomes`](https://www.ebi.ac.uk/metagenomics/api/v2/#/Miscellaneous/list_mgnify_biomes)
2. Get and initiate the `mgni_py.Client` instance
3. (Optional) add type annotations

In [12]:
# at minimum need
# 1. the path
from mgnipy.emgapi_v2_client.api.miscellaneous import list_mgnify_biomes
# 2. the client
from mgnipy.emgapi_v2_client import Client

# extra nice to have annotations but not required for usage
# 3. the models
from mgnipy.emgapi_v2_client.models import NinjaPaginationResponseSchemaBiome
from mgnipy.emgapi_v2_client.types import UNSET, Response

## Client and request handling

To instantiate the python client we really only need the base_url. However there are options for loggiing and other httpx args.

`mgnipy.emgapi_v2_client.Client`/`AuheticatedClient` will take care of constructing and closing the httpx clients

In [13]:
example_client = Client(base_url="https://www.ebi.ac.uk")
# check it out
print(example_client)

Client(raise_on_unexpected_status=False, _base_url='https://www.ebi.ac.uk', _cookies={}, _headers={}, _timeout=None, _verify_ssl=True, _follow_redirects=False, _httpx_args={}, _client=None, _async_client=None)


The get request is made when running the `.sync...()` or `.async...()` functions. For example if executing

```python
with example_client as client:
    response = list_mgnify_biomes.sync_detailed(
        client=client, page_size=10
    )
```
the order of the methods within `list_mgnify_biomes.py` is:

1. `_get_kwargs` to prepare the query params, ensuring that the kwarg exists e.g. that `page_size` is an acceptable kwarg
2. the httpx request is made with the kwargs
3. `_build_response` to prepare as `Response` type
4. In the `Response` is attribute `Response.parsed` which uses `_parse_response` to get response as json / dict

The difference between with and sans `..._detailed()` is that with returns the whole response and sans only returns the parsed response.

-----

We will now make carry out the example request to start at biome_lineage "root:Host-associated:Plants".

In [14]:
# prep our search
params = {
    "biome_lineage": "root:Host-associated:Plants",
    # this is default num of results per page in mgnify
    "page_size": 25,
    "max_depth": 6,
}

# make the sync call and store respone
with example_client as client:
    response: Response[NinjaPaginationResponseSchemaBiome] = list_mgnify_biomes.sync_detailed(client=client, **params)

# check
response.status_code

<HTTPStatus.OK: 200>

## Response handling

and if we take a look at the parsed content:

In [15]:
response.parsed.to_dict().keys()

dict_keys(['count', 'items'])

- In `items` are records such as the biome_lineages
    - for the first page of results only
- In `count` is the total number of records that matched our query.
**Note: you need to instantiate a new client object every call**

Let's take a closer look at the response

In [16]:
response.parsed.to_dict()

{'count': 15,
 'items': [{'biome_name': 'Plants', 'lineage': 'root:Host-associated:Plants'},
  {'biome_name': 'Phylloplane',
   'lineage': 'root:Host-associated:Plants:Phylloplane'},
  {'biome_name': 'Endophytes',
   'lineage': 'root:Host-associated:Plants:Phylloplane:Endophytes'},
  {'biome_name': 'Epiphytes',
   'lineage': 'root:Host-associated:Plants:Phylloplane:Epiphytes'},
  {'biome_name': 'Rhizome', 'lineage': 'root:Host-associated:Plants:Rhizome'},
  {'biome_name': 'Epiphytes',
   'lineage': 'root:Host-associated:Plants:Rhizome:Epiphytes'},
  {'biome_name': 'Rhizoplane',
   'lineage': 'root:Host-associated:Plants:Rhizoplane'},
  {'biome_name': 'Endophytes',
   'lineage': 'root:Host-associated:Plants:Rhizoplane:Endophytes'},
  {'biome_name': 'Epiphytes',
   'lineage': 'root:Host-associated:Plants:Rhizoplane:Epiphytes'},
  {'biome_name': 'Soil',
   'lineage': 'root:Host-associated:Plants:Rhizoplane:Soil'},
  {'biome_name': 'Rhizosphere',
   'lineage': 'root:Host-associated:Plants:

## Async requests support 

In [23]:
import asyncio 

example_client = Client(base_url="https://www.ebi.ac.uk")

async with example_client as client: 
    parsed_response = await list_mgnify_biomes.asyncio(
        client=client, 
        biome_lineage="root:Host-associated",
        max_depth=6,
        page_size=5,
    )

parsed_response.to_dict()

{'count': 233,
 'items': [{'biome_name': 'Host-associated',
   'lineage': 'root:Host-associated'},
  {'biome_name': 'Algae', 'lineage': 'root:Host-associated:Algae'},
  {'biome_name': 'Brown Algae',
   'lineage': 'root:Host-associated:Algae:Brown Algae'},
  {'biome_name': 'Green algae',
   'lineage': 'root:Host-associated:Algae:Green algae'},
  {'biome_name': 'Ectosymbionts',
   'lineage': 'root:Host-associated:Algae:Green algae:Ectosymbionts'}]}